In [30]:
import pickle as pkl
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import brown
from nltk import download
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import random
from tqdm import tqdm
import json
import numpy as np

torch.manual_seed(42)

In [46]:
import importlib
import enc_dec_lstm
importlib.reload(enc_dec_lstm)
from enc_dec_lstm import Encoder_Decoder_Model

In [32]:
if torch.cuda.is_available():
	device = torch.device("cuda")
elif torch.xpu.is_available():
	device = torch.device("xpu")
else:
	device = torch.device("cpu")
device

device(type='cpu')

In [33]:
embed_dim = 300

In [34]:
def load_glove(path):
    embeddings_index = {}
    with open(path, encoding="utf8") as f:
        for i, line in tqdm(enumerate(f)):
            values = line.strip().split()
            word = " ".join(values[:-embed_dim])
            vector = np.asarray(values[-embed_dim:], dtype="float32")
            vector = torch.from_numpy(vector)
            embeddings_index[word] = vector
    print(f"Loaded {len(embeddings_index)} word vectors from GloVe.")
    return embeddings_index

glove_path = "glove.2024.wikigiga.300d.txt"
glove_vectors = load_glove(glove_path)

1291147it [01:28, 14550.71it/s]


Loaded 1291147 word vectors from GloVe.


In [35]:
with open('../data/train_data.pkl', 'rb') as f:
    train_data = pkl.load(f)

with open('../data/val_data.pkl', 'rb') as f:
    val_data = pkl.load(f)

In [36]:
word_counts = Counter(w for sent in train_data for w, _ in sent)
tag_counts = Counter(t for sent in train_data for _, t in sent)

word2idx = {w: i+2 for i, (w, _) in enumerate((w1, c) for w1, c in word_counts.items())}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
idx2word = {i: w for w, i in word2idx.items()}

tag2idx = {t: i+1 for i, (t, _) in enumerate(tag_counts.items())}
tag2idx["<PAD>"] = 0
idx2tag = {i: t for t, i in tag2idx.items()}

In [37]:
train_embeddings = torch.zeros((len(word2idx), embed_dim))
for word, i in word2idx.items():
    if word=="<PAD>":
        train_embeddings[i] = torch.zeros((embed_dim,))
    elif word=="<UNK>":
        train_embeddings[i] = glove_vectors["<unk>"]
    elif word in glove_vectors:
        train_embeddings[i] = glove_vectors[word]
    else:
        train_embeddings[i] = glove_vectors["<unk>"]

In [38]:
# with open('tokenizer/word2idx.json', 'r') as f:
#     word2idx = json.load(f)
#     idx2word = {v: k for k, v in word2idx.items()}
# with open('tokenizer/tag2idx.json', 'r') as f:
#     tag2idx = json.load(f)
#     idx2tag = {v: k for k, v in tag2idx.items()}
# with open("tokenizer/word_embeddings.pt", "rb") as f:
#     train_embeddings = torch.load(f)

In [39]:
with open("tokenizer/word2idx.json", "w") as f:
    json.dump(word2idx, f)
with open("tokenizer/tag2idx.json", "w") as f:
    json.dump(tag2idx, f)
with open("tokenizer/word_embeddings.pt", "wb") as f:
    torch.save(train_embeddings, f)

In [40]:
vocab_size = len(word2idx)
tag_size = len(tag2idx)

In [41]:
class POSTagDataset(Dataset):
    def __init__(self, sentences):
        self.data = []
        for sent in sentences:
            words, tags = zip(*sent)
            
            word_ids = [word2idx.get(w, 1) for w in words]
            tag_ids = [tag2idx.get(t, 0) if t in tag2idx else 0 for t in tags]

            length = len(word_ids)

            self.data.append((torch.tensor(word_ids).to(device), torch.tensor(tag_ids).to(device), length))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [42]:
def data_collate_fn(batch):
    words, tags, lens = zip(*batch)
    words_batch = nn.utils.rnn.pad_sequence(words, batch_first=True, padding_value=0).to(device)
    tags_batch = nn.utils.rnn.pad_sequence(tags, batch_first=True, padding_value=0).to(device)
    return words_batch, tags_batch, lens

In [43]:
train_dataset = POSTagDataset(train_data)
val_dataset = POSTagDataset(val_data)

In [44]:
batch_size=128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collate_fn)

In [47]:
epochs = 100
hidden_dim = 256
model=Encoder_Decoder_Model(vocab_size, embed_dim, hidden_dim, tag_size, train_embeddings).to(device)
lr=0.001
optimizer = optim.Adam(model.parameters(), lr=lr)
# criterion = nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss(ignore_index=0)
clip = 1.0

loss_file = open("loss.txt", "w")

for e in range(epochs):
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {e+1}/{epochs}", total=len(train_loader)):
        words_batch, tags_batch, length_batch = batch
        input_seq = words_batch
        output_tags = tags_batch
        outputs = model(input_seq)
        loss = criterion(outputs.view(-1, tag_size), output_tags.view(-1))
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
    print(f"Training Loss: {train_loss / len(train_loader)}")
    loss_file.write(f"Train loss for epoch {e+1}: {train_loss / len(train_loader)}\n")

    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        for batch in tqdm(val_loader, desc=f"Validation {e+1}/{epochs}", total=len(val_loader)):
            words_batch, tags_batch, length_batch = batch
            input_seq = words_batch
            output_tags = tags_batch
            outputs = model(input_seq)
            loss = criterion(outputs.view(-1, tag_size), output_tags.view(-1))
            val_loss += loss.item()
        print(f"Validation Loss: {val_loss / len(val_loader)}")
        loss_file.write(f"Validation loss for epoch {e+1}: {val_loss / len(val_loader)}\n")

    model_path = f'models/encoder_decoder_model_{lr}lr_{batch_size}bs_{e+1}epochs.pth'
    loss_file.flush()
    if (e+1) % 5 == 0:
        print(f"Saving model at epoch {e+1} to {model_path}")
        torch.save(model.state_dict(), model_path)

loss_file.close()

Epoch 1/100: 100%|██████████| 359/359 [01:49<00:00,  3.27it/s]


Training Loss: 0.5838447218153802


Validation 1/100: 100%|██████████| 45/45 [00:04<00:00,  9.55it/s]


Validation Loss: 0.2848622613483005


Epoch 2/100: 100%|██████████| 359/359 [01:25<00:00,  4.22it/s]


...

Training Loss: 0.008843549570680951


Validation 39/100: 100%|██████████| 45/45 [00:13<00:00,  3.24it/s]


Validation Loss: 0.24977083537313674


Epoch 40/100:  91%|█████████▏| 328/359 [20:04<01:53,  3.67s/it]    


KeyboardInterrupt: 